# 04 - Adapted Volatility-of-Volatility Methods

Construct the NEPSE benchmark and adapted Yuan-style measures.


In [1]:
import numpy as np
import pandas as pd

returns_df = pd.read_csv("data_nepse_returns.csv", parse_dates=["date_ad"], index_col="date_ad")
garch = pd.read_csv("data_garch_volatility_estimates.csv", parse_dates=["date_ad"], index_col="date_ad")

df = returns_df.join(garch, how="left")
for window in [10, 30, 60]:
    df[f"hv_{window}d_daily_pct"] = df["log_return"].rolling(window).std() * 100
    df[f"hv_{window}d_annualized_pct"] = df[f"hv_{window}d_daily_pct"] * np.sqrt(252)

df.head()


,index_value,absolute_change,percentage_change,calendar_gap_days,is_long_gap,log_return,log_return_pct,rv_30d_annualized_pct,garch_11_daily_pct,garch_11_annualized_pct,egarch_11_daily_pct,egarch_11_annualized_pct,gjr_garch_11_daily_pct,gjr_garch_11_annualized_pct,hv_10d_daily_pct,hv_10d_annualized_pct,hv_30d_daily_pct,hv_30d_annualized_pct,hv_60d_daily_pct,hv_60d_annualized_pct
date_ad,,,,,,,,,,,,,,,,,,,,
2015-01-01,903.68,1.35,0.15,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-04,917.50,13.82,1.53,3.0,False,0.015177,1.517726,NaN,0.916569,14.550080,0.910679,14.456584,0.916096,14.542570,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,919.77,2.27,0.25,1.0,False,0.002471,0.247106,NaN,1.073247,17.037264,1.096887,17.412536,1.029692,16.345858,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-06,917.41,-2.36,-0.26,1.0,False,-0.002569,-0.256916,NaN,0.996248,15.814951,1.001278,15.894803,0.962316,15.276300,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-07,924.54,7.13,0.78,1.0,False,0.007742,0.774183,NaN,0.938972,14.905712,0.935263,14.846839,0.913990,14.509137,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Benchmark: realized volatility-of-volatility from 30-day historical volatility.
df["benchmark_vov_30d"] = df["hv_30d_daily_pct"].rolling(30).var()

# STVV: historical variance of return variance.
df["stvv_historical"] = (df["log_return_pct"] ** 2).rolling(30).var()

# EMVV-adapted: substitute NEPSE realized volatility changes for VIX log-return inputs.
df["emvv_adapted"] = np.log(df["hv_30d_daily_pct"] / df["hv_30d_daily_pct"].shift(1)).rolling(30).var()

# MIVV-adapted: substitute GARCH conditional volatility for option-implied volatility.
df["mivv_adapted_garch"] = np.log(
    df["garch_11_daily_pct"] / df["garch_11_daily_pct"].shift(1)
).rolling(30).var()
df["mivv_adapted_egarch"] = np.log(
    df["egarch_11_daily_pct"] / df["egarch_11_daily_pct"].shift(1)
).rolling(30).var()

method_cols = [
    "benchmark_vov_30d",
    "stvv_historical",
    "emvv_adapted",
    "mivv_adapted_garch",
    "mivv_adapted_egarch",
]
df[method_cols].dropna().head()


,benchmark_vov_30d,stvv_historical,emvv_adapted,mivv_adapted_garch,mivv_adapted_egarch
date_ad,,,,,
2015-04-05,0.016857,0.149182,0.004350,0.003294,0.006638
2015-04-06,0.016214,0.149479,0.004361,0.003305,0.006696
2015-04-07,0.015117,0.117279,0.004296,0.003327,0.006686
2015-04-08,0.013823,0.116691,0.004363,0.002381,0.005272
2015-04-09,0.012376,0.115381,0.004389,0.002333,0.005263


In [3]:
df.to_csv("data_vov_methods.csv")
print("Saved adapted volatility-of-volatility methods to data_vov_methods.csv")
df[method_cols].dropna().tail()


Saved adapted volatility-of-volatility methods to data_vov_methods.csv


,benchmark_vov_30d,stvv_historical,emvv_adapted,mivv_adapted_garch,mivv_adapted_egarch
date_ad,,,,,
2026-05-26,0.119240,0.406894,0.003738,0.004947,0.008694
2026-05-27,0.123951,0.406533,0.003740,0.004706,0.008368
2026-06-01,0.126456,0.387690,0.003701,0.004455,0.007990
2026-06-02,0.125564,0.359475,0.003652,0.004762,0.008466
2026-06-03,0.121607,0.360314,0.003649,0.004773,0.008521
